In [ ]:
# Import necessary libraries
import json
import os
from typing import Any, Dict

import boto3
import requests
from dotenv import load_dotenv

load_dotenv()
print("We are up and running!")

We are up and running!


AWS credentials are picked up automatically from the environment (instance profile, env vars, or `~/.aws/credentials`).

In [11]:
MODEL_ID = "eu.amazon.nova-lite-v1:0"
REGION = "eu-west-1"

bedrock = boto3.client("bedrock-runtime", region_name=REGION)

print(f"Bedrock client ready — model: {MODEL_ID}")

Bedrock client ready — model: eu.amazon.nova-lite-v1:0


In [12]:
PROMPT = "What is the current price of Bitcoin?"

response = bedrock.converse(
    modelId=MODEL_ID,
    messages=[
        {"role": "user", "content": [{"text": PROMPT}]}
    ],
)

print(response["output"]["message"]["content"][0]["text"])

I'm unable to provide real-time data, including the current price of Bitcoin, as my information is not updated in real time and I do not have access to external sources of data at the moment. However, you can easily find the current price of Bitcoin by checking reputable financial news websites, cryptocurrency exchanges, or using financial apps that track cryptocurrency prices. Some popular exchanges where you can check the current price include Coinbase, Binance, and Kraken, among others.


In [13]:
# Inspect the full raw response from Bedrock
print(json.dumps(response, indent=2, default=str))

{
  "ResponseMetadata": {
    "RequestId": "f62d1d2f-0309-4675-a3f5-81336d039560",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Mon, 02 Mar 2026 12:22:12 GMT",
      "content-type": "application/json",
      "content-length": "696",
      "connection": "keep-alive",
      "x-amzn-requestid": "f62d1d2f-0309-4675-a3f5-81336d039560"
    },
    "RetryAttempts": 0
  },
  "output": {
    "message": {
      "role": "assistant",
      "content": [
        {
          "text": "I'm unable to provide real-time data, including the current price of Bitcoin, as my information is not updated in real time and I do not have access to external sources of data at the moment. However, you can easily find the current price of Bitcoin by checking reputable financial news websites, cryptocurrency exchanges, or using financial apps that track cryptocurrency prices. Some popular exchanges where you can check the current price include Coinbase, Binance, and Kraken, among others."
        }
   

In [14]:
url = "https://api.binance.com/api/v3/ticker/price?symbol=BTCUSDT"
response = requests.get(url)
data = response.json()
print(data)

{'symbol': 'BTCUSDT', 'price': '66169.67000000'}


In [15]:
# Define the function
def get_crypto_price(symbol: str) -> float:
    """
    Get the current price of a cryptocurrency from Binance API
    """
    url = f"https://api.binance.com/api/v3/ticker/price?symbol={symbol}"
    response = requests.get(url)
    data = response.json()
    return float(data["price"])

In [16]:
price = get_crypto_price("BTCUSDT")
print(f"BTC Price in USDT: {price}")

BTC Price in USDT: 66169.68


In [17]:
# Tool definition in Bedrock Converse format
tool_config = {
    "tools": [
        {
            "toolSpec": {
                "name": "get_crypto_price",
                "description": "Get cryptocurrency price in USDT from Binance",
                "inputSchema": {
                    "json": {
                        "type": "object",
                        "properties": {
                            "symbol": {
                                "type": "string",
                                "description": (
                                    "The cryptocurrency trading pair symbol "
                                    "(e.g., BTCUSDT, ETHUSDT). "
                                    "The symbol for Bitcoin is BTCUSDT. "
                                    "The symbol for Ethereum is ETHUSDT."
                                ),
                            }
                        },
                        "required": ["symbol"],
                    }
                },
            }
        }
    ]
}

In [18]:
PROMPT = "What is the current price of Bitcoin?"

messages = [
    {"role": "user", "content": [{"text": PROMPT}]}
]

response = bedrock.converse(
    modelId=MODEL_ID,
    messages=messages,
    toolConfig=tool_config,
)

# Print the full raw response so we can see the tool call request
print(json.dumps(response, indent=2, default=str))

{
  "ResponseMetadata": {
    "RequestId": "d5fdab80-2cd8-4aa8-9781-9f6a30320f7e",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Mon, 02 Mar 2026 12:22:14 GMT",
      "content-type": "application/json",
      "content-length": "499",
      "connection": "keep-alive",
      "x-amzn-requestid": "d5fdab80-2cd8-4aa8-9781-9f6a30320f7e"
    },
    "RetryAttempts": 0
  },
  "output": {
    "message": {
      "role": "assistant",
      "content": [
        {
          "text": "<thinking> The User wants to know the current price of Bitcoin. I need to use the \"get_crypto_price\" tool to get this information. The symbol for Bitcoin is BTCUSDT. </thinking>\n"
        },
        {
          "toolUse": {
            "toolUseId": "tooluse_8W8MljrQH2hsYauY9MoHlu",
            "name": "get_crypto_price",
            "input": {
              "symbol": "BTCUSDT"
            }
          }
        }
      ]
    }
  },
  "stopReason": "tool_use",
  "usage": {
    "inputTokens": 444,
    "o

In [19]:
# Extract the tool call from the response
assistant_message = response["output"]["message"]
tool_use_block = next(
    block for block in assistant_message["content"] if "toolUse" in block
)

tool_use_id = tool_use_block["toolUse"]["toolUseId"]
tool_name = tool_use_block["toolUse"]["name"]
tool_input = tool_use_block["toolUse"]["input"]

print(f"Tool requested: {tool_name}")
print(f"Tool input:     {tool_input}")
print(f"Tool use ID:    {tool_use_id}")

Tool requested: get_crypto_price
Tool input:     {'symbol': 'BTCUSDT'}
Tool use ID:    tooluse_8W8MljrQH2hsYauY9MoHlu


In [20]:
# Manually call the function with the arguments the model provided
price = get_crypto_price(**tool_input)
print(f"Result from function: {price}")

Result from function: 66176.53


In [21]:
# Build the full conversation: original user message + assistant tool call + our tool result
messages = [
    {"role": "user", "content": [{"text": PROMPT}]},
    assistant_message,  # the model's response containing the toolUse block
    {
        "role": "user",
        "content": [
            {
                "toolResult": {
                    "toolUseId": tool_use_id,
                    "content": [{"text": str(price)}],
                }
            }
        ],
    },
]

final_response = bedrock.converse(
    modelId=MODEL_ID,
    messages=messages,
    toolConfig=tool_config,
)

print(json.dumps(final_response, indent=2, default=str))

{
  "ResponseMetadata": {
    "RequestId": "a2c35f6d-71fe-4450-8b44-4504ab0ae083",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Mon, 02 Mar 2026 12:22:16 GMT",
      "content-type": "application/json",
      "content-length": "458",
      "connection": "keep-alive",
      "x-amzn-requestid": "a2c35f6d-71fe-4450-8b44-4504ab0ae083"
    },
    "RetryAttempts": 0
  },
  "output": {
    "message": {
      "role": "assistant",
      "content": [
        {
          "text": "<thinking> I have used the \"get_crypto_price\" tool and obtained the result. The current price of Bitcoin (BTCUSDT) is 66176.53 USDT. I can now provide this information to the User. </thinking>\n\nThe current price of Bitcoin (BTCUSDT) is 66176.53 USDT."
        }
      ]
    }
  },
  "stopReason": "end_turn",
  "usage": {
    "inputTokens": 545,
    "outputTokens": 79,
    "totalTokens": 624
  },
  "metrics": {
    "latencyMs": 736
  }
}


In [22]:
import re

text = final_response["output"]["message"]["content"][0]["text"]
clean_text = re.sub(r"<thinking>.*?</thinking>", "", text, flags=re.DOTALL | re.IGNORECASE).strip()

print(clean_text)

The current price of Bitcoin (BTCUSDT) is 66176.53 USDT.
